# Orbit Wars — RL Observation & Planet-Only Set Transformer Policy

Bu defter, repo'daki güncel RL observation ve policy akışını uçtan uca açıklar:

- **`obs_builder.py`** — Ham env observation'ını sabit şekilli numpy tensörlerine çeviren feature pipeline.
- **`model_planet_only.py`** — Fleet token kullanmadan, planet + global token üzerinden çalışan güncel default PPO policy.
- **`model.py`** — İstenirse `--model-variant full` ile kullanılabilen eski/full Set Transformer policy.

## Güncel mimari kararı

Yeni default akışta havadaki filolar tek tek model token'ı olarak verilmez. Bunun yerine her fleet, tahmin edilen hedef gezegeninin planet feature'larına ETA bucket aggregate olarak işlenir.

| Özellik | Tasarıma etkisi |
| --- | --- |
| Değişken sayıda gezegen | Sabit padding + mask |
| Çok sayıda fleet | Fleet token yerine hedef planet üzerinde aggregate ETA bucket |
| Permutation invariance | Planet set encoder |
| Aksiyon = `[from_id, angle, ships]` | Per-planet pointer/action head |
| Geometrik mantık kritik | Edge feature → attention bias |

**Default kararımız**: `PlanetOnlyOrbitWarsPolicy` + edge bias + masked pointer policy. Fleet bilgisi planet token'larının son 8 kanalına ve global feature'lara girer.

In [ ]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
sys.path.insert(0, REPO)
print('Repo root:', REPO)

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from kaggle_environments import make

from feature_extraction import parse_planets, parse_fleets, SUN_X, SUN_Y, SUN_RADIUS
from obs_builder import (
    build_observation, decode_action,
    PLANET_FEATURE_DIM, FLEET_FEATURE_DIM, GLOBAL_FEATURE_DIM, EDGE_FEATURE_DIM,
)
from model import OrbitWarsPolicy, pack_observations
from model_planet_only import PlanetOnlyOrbitWarsPolicy
import torch

## 1. Bir oyun başlat ve raw observation al

In [ ]:
env = make('orbit_wars', debug=False)
env.reset(2)
# `env.reset` allocates the env but doesn't generate the world; the first
# `env.step` populates planets, comets, and angular_velocity.
env.step([[], []])

obs0 = env.state[0].observation  # player 0 perspective
omega = obs0.get('angular_velocity', 0.0)
print(f'#planets = {len(obs0["planets"])}, #fleets = {len(obs0["fleets"])}, omega = {omega:.4f}')
print('Sample planet row:', obs0['planets'][0])
print('Observation keys:', list(obs0.keys()))

Ham observation `dict`:

- `planets`: `[id, owner, x, y, radius, ships, production]` listesi
- `fleets`: `[id, owner, x, y, angle, from_planet_id, ships]` listesi (havadaki filolar)
- `initial_planets`: oyun başındaki gezegen konumları (rotating tipini belirlemek için)
- `comet_planet_ids`, `comets`: kuyruklu yıldız bilgileri
- `angular_velocity`: rotating gezegenlerin açısal hızı (sabit, oyuna özel)

Bu yapıyı doğrudan ağa veremeyiz: değişken uzunluk, karışık tipler, ölçeksiz değerler.

## 2. `build_observation` ile feature pipeline

Tek çağrıda raw obs → padded numpy tensörleri.

In [ ]:
o = build_observation(obs0, player=0, angular_velocity=omega, turn=0, f_max=0)
print(f'planet_feats : {o.planet_feats.shape}  ({PLANET_FEATURE_DIM} feats)')
print(f'planet_mask  : {o.planet_mask.shape}  -> real planets = {int(o.planet_mask.sum())}')
print(f'source_mask  : {o.source_mask.shape}  -> launchable   = {int(o.source_mask.sum())}')
print(f'fleet_feats  : {o.fleet_feats.shape}  ({FLEET_FEATURE_DIM} feats; f_max=0 in planet-only default)')
print(f'fleet_mask   : {o.fleet_mask.shape}   -> real fleet tokens = {int(o.fleet_mask.sum())}')
print(f'global_feats : {o.global_feats.shape}  ({GLOBAL_FEATURE_DIM} feats)')
print(f'edge_bias    : {o.edge_bias.shape}  -> T = P_max + f_max + 1 = {o.edge_bias.shape[0]}')
print(f'meta         : {o.meta}')

### 2.1 Planet feature'ları (31-d)

Planet token artık sadece gezegenin statik/dinamik özelliklerini değil, o gezegene doğru gelen filo özetlerini de taşır. Güncel default modelde fleet token yoktur; bu yüzden fleet bilgisi planet feature'larının incoming ETA bucket ve aggregate threat kanallarına taşınır.

| idx | feature | açıklama |
| --- | --- | --- |
| 0–2 | `is_self, is_enemy, is_neutral` | sahip one-hot |
| 3 | `ships / 100` | gezegen üstündeki mevcut gemi sayısı |
| 4 | `production / 5` | üretim normalize |
| 5 | `radius / 5` | gezegen yarıçapı |
| 6 | `r_sun / 50` | güneşe uzaklık |
| 7,8 | `cos(theta), sin(theta)` | güneşe göre polar açı |
| 9–11 | `is_static, is_rotating, is_comet` | gezegen tipi |
| 12 | `omega * dt` | rotating planet için açısal hız sinyali |
| 13 | `(ships+1) / 100` | ele geçirmek için kaba minimum gemi ihtiyacı; self planet için 0 |
| 14 | `log(ships+1) / log(1000)` | büyük gemi sayılarını daha yumuşak ölçekler |
| 15–18 | `incoming_self_* / 500` | current player filosu ETA bucket'ları: `0–5`, `5–15`, `15–30`, `30+` |
| 19–22 | `incoming_enemy_* / 500` | enemy filo ETA bucket'ları: `0–5`, `5–15`, `15–30`, `30+` |
| 23 | `incoming_self_total / 500` | toplam self incoming |
| 24 | `incoming_enemy_total / 500` | toplam enemy incoming |
| 25 | `incoming_self_min_eta / 50` | en yakın self incoming ETA |
| 26 | `incoming_enemy_min_eta / 50` | en yakın enemy incoming ETA |
| 27 | `incoming_enemy_soonest_ships / 500` | en erken enemy incoming gemisi |
| 28 | `incoming_enemy_largest_ships / 500` | en büyük enemy incoming gemisi |
| 29 | `incoming_enemy_largest_eta / 50` | en büyük enemy incoming ETA |
| 30 | `incoming_net_threat / 500` | `enemy_total - ships - self_total` |

Bucket sınırları kodda şu sabitle tanımlıdır:

```python
INCOMING_ETA_BUCKETS = (5.0, 15.0, 30.0, float('inf'))
```

Bu bilgi target planet'in owner'ına göre değil, **filonun owner'ının current player'a göre self/enemy olmasına** göre sınıflanır. Örneğin player 0, player 1'e ait bir gezegene saldırı gönderirse bu target üzerinde `incoming_self_*` bucket'ı artar; çünkü bu current player'ın kendi planıdır.

Polar koordinatlar (`r_sun`, `cos/sin θ`) ham `(x, y)` yerine tercih edildi: Orbit Wars'ın simetrisi rotational, polar'da öğrenmek daha kolay.

In [ ]:
import pandas as pd
cols = [
    'is_self','is_enemy','is_neutral','ships/100','prod/5','radius/5',
    'r_sun/50','cos(theta)','sin(theta)','is_static','is_rotating',
    'is_comet','omega*dt','ships_to_cap','log_ships',
    'incoming_self_0_5','incoming_self_5_15',
    'incoming_self_15_30','incoming_self_30_plus',
    'incoming_enemy_0_5','incoming_enemy_5_15',
    'incoming_enemy_15_30','incoming_enemy_30_plus',
    'incoming_self_total','incoming_enemy_total',
    'incoming_self_min_eta','incoming_enemy_min_eta',
    'incoming_enemy_soonest_ships','incoming_enemy_largest_ships',
    'incoming_enemy_largest_eta','incoming_net_threat',
]
n = int(o.planet_mask.sum())
df = pd.DataFrame(o.planet_feats[:n], columns=cols)
df.insert(0, 'planet_id', o.planet_ids[:n])
df.round(3)

### 2.1.1 Incoming fleet aggregate'leri neden planet feature içinde?

Fleet token'ları havadaki filoları tek tek temsil edebilir; fakat güncel default modelde fleet token kaldırıldı. Karar verirken çoğu zaman şu soru daha doğrudan önemlidir:

```text
Bu gezegene kaç gemi geliyor, benden mi düşmandan mı geliyor, hangi ETA aralığında geliyor?
```

Bu yüzden her planet token'a 8 incoming bucket kanalı eklendi:

```text
self incoming:  current player'ın bu gezegene yolladığı filolar
enemy incoming: current player dışındaki tüm owner'ların bu gezegene yolladığı filolar
ETA bucket'ları: 0-5, 5-15, 15-30, 30+ turn
```

Enemy tarafı artık `enemy_0/1/2` owner slotlarına ayrılmaz; tüm enemy filolar aynı ETA bucket aggregate'lerinde toplanır. Böylece sınırlı enemy slot kapasitesi yerine hedef gezegendeki zaman dağılımı daha net görünür.

Bu ayrım multi-action policy için kritik. Aynı env turn içinde sırasıyla birden fazla source kararı alacaksak, ilk launch'tan sonra geçici observation şöyle güncellenir:

```text
source planet ships                    -= sent_ships
target incoming_self_bucket[launch_eta] += sent_ships
source_mask[source]                    = False
```

Böylece ikinci source kararı, birinci source'un target'a hangi ETA aralığında kaç gemi yolladığını observation üzerinden görebilir.

### 2.2 Fleet feature'ları (11-d) — sadece full variant için

Güncel default `planet_only` modelde `f_max=0` olduğu için fleet token yoktur. Yine de `model.py` içindeki eski/full `OrbitWarsPolicy` variant'ı kullanılmak istenirse fleet feature pipeline hâlâ durur.

| idx | feature |
| --- | --- |
| 0–1 | `is_self, is_enemy` |
| 2 | `ships / 100` |
| 3 | `log_ships` |
| 4 | `speed / max_speed` |
| 5,6 | `cos(angle), sin(angle)` |
| 7,8 | `x/100 - 0.5, y/100 - 0.5` |
| 9 | `predicted_eta / 50` (`navigation.predict_fleet_target` ile) |
| 10 | `has_predicted_target` |

Planet-only default'ta bu bilgi tek tek token olarak değil, hedef planet üstündeki ETA bucket aggregate'leri ve global fleet summary üzerinden kullanılır.

In [ ]:
if o.fleet_mask.any():
    fcols = ['is_self','is_enemy','ships/100','log_ships','speed/max',
             'cos(a)','sin(a)','x_c','y_c','eta/50','has_pred']
    nf = int(o.fleet_mask.sum())
    fdf = pd.DataFrame(o.fleet_feats[:nf], columns=fcols)
    fdf.insert(0, 'fleet_id', o.fleet_ids[:nf])
    print(fdf.round(3))
else:
    print('Henüz havada filo yok — env\'i birkaç adım ilerletip tekrar bakabiliriz.')

### 2.3 Global feature'lar (12-d)

Bütün state'in özet istatistiği. Transformer içinde **CLS-style global token** olarak yer alır.

In [ ]:
gcols = ['turn/200','omega','my_planets/25','enemy_planets/25','neutral/25',
        'my_ships/1000','enemy_ships/1000','my_prod/50','enemy_prod/50',
        'fleets/25','incoming_enemy/500','incoming_friendly/500']
for name, val in zip(gcols, o.global_feats):
    print(f'  {name:<22s} = {val:+.3f}')

### 2.4 Edge bias matrisi (T × T × 5)

Token sırası: `[planet_0, ..., planet_{P-1}, fleet_0, ..., fleet_{F-1}, GLOBAL]`. Her çift için 5 feature:

- `dist / 100` — Öklid mesafesi (normalize)
- `dx / 100` — işaretli x farkı (yön)
- `dy / 100` — işaretli y farkı (yön)
- `sin(Δφ)` — güneş etrafındaki açı farkının sinüsü
- `cos(Δφ)` — güneş etrafındaki açı farkının kosinüsü

Son iki kanal, her token'ın güneşe göre polar açısı `φ = atan2(y − sun_y, x − sun_x)` kullanılarak hesaplanır. İki gezegen **aynı** `angular_velocity` ile döndüğü için aralarındaki `Δφ` zamanla değişmez — model bu sayede board rotasyonundan bağımsız, kalıcı bir orbital ilişki sinyali alır. `(dx, dy)` her turn döndüğü için anlık geometriyi, `(sin Δφ, cos Δφ)` ise zamandan bağımsız orbital ilişkiyi taşır.

Bu matris transformer'ın attention skorlarına **per-head additive bias** olarak eklenir; model, mesafeyi ve orbital yapıyı sıfırdan öğrenmek zorunda kalmaz.

In [ ]:
n = int(o.planet_mask.sum())
channel_names = ['dist/100', 'dx/100', 'dy/100', 'sin(dphi)', 'cos(dphi)']
fig, ax = plt.subplots(1, 5, figsize=(20, 4))
for idx, name in enumerate(channel_names):
    im = ax[idx].imshow(o.edge_bias[:n, :n, idx], cmap='RdBu_r', vmin=-1, vmax=1)
    ax[idx].set_title(f'edge_bias[..., {idx}] = {name}')
    ax[idx].set_xlabel('target planet'); ax[idx].set_ylabel('source planet')
    plt.colorbar(im, ax=ax[idx], fraction=0.046)
plt.tight_layout(); plt.show()

### 2.5 Maskeleme semantiği

Üç ayrı mask:

- **`planet_mask`**: gerçek gezegen mi, yoksa padding mi? Transformer attention bunları görmez.
- **`fleet_mask`**: aynısı filo için.
- **`source_mask`**: yalnızca **kendi gezegenlerimiz + yeterli gemisi olanlar** kaynak olabilir. Policy'nin `src` head'inde bu mask uygulanıyor → yasadışı kaynaklara olasılık sızmıyor.

In [ ]:
n = int(o.planet_mask.sum())
owners = o.planet_feats[:n, :3].argmax(axis=1)  # 0=self, 1=enemy, 2=neutral
owner_label = {0:'self', 1:'enemy', 2:'neutral'}
print(f'{"slot":<5}{"id":<5}{"owner":<10}{"ships":<8}{"src_mask":<10}')
for i in range(n):
    ships = o.planet_feats[i, 3] * 100
    print(f'{i:<5d}{o.planet_ids[i]:<5d}{owner_label[owners[i]]:<10s}'
          f'{int(ships):<8d}{str(bool(o.source_mask[i])):<10s}')

## 3. Planet-only Set Transformer policy (`PlanetOnlyOrbitWarsPolicy`)

### 3.1 Token akışı

Güncel default PPO variant'ı fleet token kullanmaz. Token dizisi sadece planet token'ları ve global token'dan oluşur:

```
         planet feats (P, 31)          global feats (12,)
                │                            │
         planet_proj                  global_proj  (her biri Linear → d_model)
                │                            │
                └────── concat ──────────────┘
                         (B, P_max + 1, d_model)
                                  │
                      + type_embed (planet/global)
                                  │
                  ┌──── n_layers × TransformerBlock ────┐
                  │  (BiasedMHA: edge_bias → +attn)     │
                  │  (FFN: d → 4d → d)                  │
                  └─────────────────────────────────────┘
                                  │
                       LayerNorm → tokens (B, T, d)
                                  │
          ┌───────────────┬───────────┴────────┬──────────────────┐
      src_head        tgt_query/tgt_key       ship_head        value_head
  (B, P) + NO-OP     (B, P, P) pointer       (B, P, k)         (B,)
```

`planet_proj` 31 feature alır. Incoming bucket ve aggregate threat kanalları, planet token'ın o gezegene gelen self/enemy filo ETA dağılımını doğrudan görmesini sağlar.

Full variant (`OrbitWarsPolicy`) hâlâ korunur ve `--model-variant full` ile fleet token'lı akış denenebilir.

### 3.2 Güncel rollout aksiyon semantiği: `act_source`

Modelde klasik `act(out)` API’si hâlâ vardır; ama PPO rollout şu anda `act_source(...)` kullanır. Yani her turn’de owned source slotları sırayla gezilir:

```text
for src_slot in source_slots:
    launch/no-launch ~ Cat([0, src_raw[src_slot]])
    if launch:
        tgt_slot ~ Cat(tgt_logits[src_slot])
        ship_bin ~ Cat(ship_logits[src_slot])
```

Tek source kararının log-prob'u:

```text
log π(a_i | s_i, src_i)
= log p(launch_i | s_i, src_i)
+ 1[launch_i] · (log p(tgt_i | s_i, src_i) + log p(ship_i | s_i, src_i))
```

PPO update sırasında aynı semantik `evaluate_source_actions(...)` ile yeniden hesaplanır. Bu yüzden batch içindeki `src`, `tgt`, `ship_bin`, `is_noop`, `old_log_prob` alanları source-conditioned karar semantiğine göredir.

### 3.3 Açıyı kim hesaplıyor?

Agent açı seçmez. Agent sadece source/target/ship/no-op kararı verir. `decode_action_with_launch(...)` ve `navigation.compute_launch(...)` bunu env action'a dönüştürür:

```text
[from_planet_id, angle_in_radians, ships]
```

`compute_launch`, hareketli hedef için intercept ETA çözer, `atan2(intercept_y - src.y, intercept_x - src.x)` ile radyan açı üretir ve güneş/gezegen çarpışması olan rotaları reddeder.

### 3.2.1 `src_head`, `tgt_query/tgt_key` ve faktörize policy detayı

Bu policy tek seferde doğrudan bütün env action'larını enumerate etmiyor. Orbit Wars aksiyonu aslında üç parçaya ayrılıyor:

```text
aksiyon = (source planet, target planet, ship fraction)
```

Kodda bu şu olasılık ayrışımıyla modelleniyor:

```text
π(a | s) = π(src | s) · π(tgt | s, src) · π(ship_bin | s, src)
```

Eğer `src = NO-OP` seçilirse target ve ship branch'leri sadece tensor shape tutarlılığı için örnekleniyor; log-prob'a katkıları sıfırlanıyor:

```text
log π(a|s) = log π(src|s)
          + 1[src != NO-OP] · (log π(tgt|s,src) + log π(ship|s,src))
```

Bu ayrışım action space'i ciddi küçültüyor. Naive joint action sayısı yaklaşık:

```text
P kaynak × P hedef × K ship_bin
```

olurdu. Burada ise model üç daha küçük dağılım üretiyor:

```text
src_logits  : (B, P+1)
tgt_logits  : (B, P, P)
ship_logits : (B, P, K)
```

---

#### `src_head`: hangi gezegenden çıkış yapılacak?

Transformer encoder'dan sonra elimizde her gezegen için bağlamlandırılmış bir token var:

```text
planet_tok: (B, P, d_model)
```

Bu token artık sadece gezegenin kendi feature'larını değil, attention sayesinde diğer gezegenleri, fleet'leri, global state'i ve edge geometry bilgisini de içerir.

`src_head` her planet token'ını tek skora indirir:

```python
src_raw = src_head(planet_tok).squeeze(-1)  # (B, P)
```

Yani her gezegen için şu soru cevaplanır:

```text
Bu state'te bu gezegenden fleet göndermek mantıklı mı?
```

Sonra `source_mask` uygulanır:

```python
src_raw = src_raw.masked_fill(~source_mask, very_neg)
```

Bunun anlamı:

- Kendi gezegenimiz değilse kaynak olamaz.
- Yeterli gemisi yoksa kaynak olamaz.
- Padding planet slot'ları kaynak olamaz.

Ardından ayrı bir öğrenilebilir `NO-OP` logit'i eklenir:

```python
src_logits = concat([src_raw, noop_logit])  # (B, P+1)
```

Son slot `P` indeksidir ve `NO-OP` anlamına gelir. Böylece policy gerektiğinde "hiçbir şey yapma" aksiyonunu da öğrenebilir.

---

#### `tgt_query` / `tgt_key`: source seçilince hedef nereye?

Target branch pointer mantığıyla çalışır. Her planet token iki farklı uzaya projekte edilir:

```python
q = tgt_query(planet_tok)  # source gibi davranacak temsil
k = tgt_key(planet_tok)    # target gibi davranacak temsil
```

Sonra her source-target çifti için dot-product skor hesaplanır:

```python
tgt_logits[i, j] = dot(q_i, k_j) / sqrt(d_model)
```

Tensor shape olarak:

```text
q          : (B, P, d)
k          : (B, P, d)
tgt_logits : (B, P, P)
```

Buradaki `tgt_logits[:, i, j]` şu anlama gelir:

```text
Eğer source olarak planet i seçildiyse, target olarak planet j ne kadar mantıklı?
```

Bu yüzden buna pointer head denebilir: model sabit class label seçmiyor; mevcut planet setindeki bir slot'a işaret ediyor.

Target skorları doğrudan raw feature'lardan değil, transformer sonrası `planet_tok` üzerinden geldiği için her skor şunlardan etkilenebilir:

- Source gezegenin gemi sayısı, owner'ı, üretimi
- Target gezegenin owner'ı, gemisi, üretimi
- Diğer gezegenlerin ve fleet'lerin durumu
- Edge bias üzerinden mesafe/yön/orbital ilişki
- Global token üzerinden oyun fazı ve genel güç dengesi

Sonra target mask uygulanır:

```python
# i -> i yasak
tgt_logits[:, i, i] = -inf

# padding target yasak
tgt_logits[:, :, padded_j] = -inf
```

Böylece model kendi kendini hedefleyemez ve gerçek olmayan padding slot'larına olasılık veremez.

---

#### Neden `tgt_logits` tüm `(P, P)` matrisi olarak üretiliyor?

Forward pass sırasında henüz hangi `src` örnekleneceği bilinmediği için model tüm olası source-target satırlarını üretir:

```text
tgt_logits[b, src_slot, :] = bu source için target dağılımı
```

Action sampling aşamasında önce `src` seçilir:

```python
src ~ Categorical(src_logits)
```

Sonra sadece seçilen source satırı alınır:

```python
tgt_logits_sel = tgt_logits.gather(source row)  # (B, P)
tgt ~ Categorical(tgt_logits_sel)
```

Yani target dağılımı source'a koşulludur:

```text
π(tgt | s, src)
```

---

#### `ship_head`: seçilen source'dan kaç gemi gönderilecek?

`ship_head` de planet token başına `K` adet skor üretir:

```python
ship_logits = ship_head(planet_tok)  # (B, P, K)
```

Buradaki `K`, discrete ship fraction bin sayısıdır:

```text
10%, 25%, 50%, 75%, 100%
```

Sampling sırasında yine sadece seçilen source satırı kullanılır:

```python
ship_logits_sel = ship_logits[src_slot]  # (K,)
ship_bin ~ Categorical(ship_logits_sel)
```

Bu da şu dağılımdır:

```text
π(ship_bin | s, src)
```

Önemli nokta: `ship_head` target'a doğrudan koşullu değil. Yani mevcut modelde gönderilecek gemi fraksiyonu source'a göre seçiliyor; target etkisi ancak transformer token'ına dolaylı olarak yansımış olabilir. Daha güçlü bir tasarımda `ship_bin` de `(src, tgt)` çiftine koşullandırılabilir.

---

#### Teorik yorum: bu neden mantıklı?

Bu yapı Orbit Wars için iyi bir inductive bias verir:

1. Önce **hareket etmeye değer bir kaynak planet** seçilir.
2. Sonra o kaynaktan **hangi planet'e gitmek iyi** sorulur.
3. Sonra o kaynaktan **ne kadar gemi göndermek gerekir** seçilir.

Bu, insan oyuncunun karar sürecine benzer. Ayrıca permutation-invariant kalır: planet slot sırası değişse bile token/head yapısı aynı mantıkla çalışır.

---

#### Mevcut sınırlama

`tgt_query/tgt_key` skoru şu anda final aşamada explicit `edge_ij` feature'ını tekrar kullanmıyor:

```text
score(i,j) = q_i · k_j
```

Geometri bilgisi encoder içinde attention bias ile token'lara yedirilmiş durumda, yani dolaylı olarak var. Ama daha relation-centric bir modelde target score şöyle zenginleştirilebilir:

```text
score(i,j) = MLP([token_i, token_j, token_i * token_j, edge_ij, action_features_ij])
```

Buradaki `action_features_ij` örnekleri:

- source-target ETA
- path reachable mı?
- sun/planet collision riski
- target arrival anındaki tahmini gemi sayısı
- gönderilen gemi / capture requirement oranı

Şu anki model öğrenebilir; ama bu ek pair/action feature'lar verilirse öğrenme daha hızlı ve daha kararlı olabilir.

### 3.2.2 Incoming bucket feature'larla sequential multi-action tasarımı

Mevcut rollout her owned source için sırayla karar üretir. Doğru olasılık ayrışımı artık tek `π(a|s)` değil, aynı turn içindeki action sequence'in çarpımıdır:

```text
A = [a_1, a_2, ..., a_m]
π(A | s_0) = ∏_i π(a_i | s_i)
```

Burada kritik nokta:

```text
s_i = önceki aynı-turn launch kararları geçici observation'a işlenmiş state
```

Bir source için karar şöyle ayrışabilir:

```text
π(a_i | s_i)
= π(launch_i | s_i, source_i)
  · [π(tgt_i | s_i, source_i) · π(ship_i | s_i, source_i)] ^ launch_i
```

Log-prob ise PPO için toplanır:

```text
log π(A | s_0)
= Σ_i log π(launch_i | s_i, source_i)
+ Σ_i 1[launch_i] · (
    log π(tgt_i | s_i, source_i)
  + log π(ship_i | s_i, source_i)
)
```

Yeni planet incoming bucket kanalları bu `s_i` güncellemesini pratik hale getirir. Örneğin `source_i -> target_j` launch edildikten sonra env step beklemeden temporary observation'da şunlar yapılır:

```text
source_i ships feature                    -= sent_ships
target_j incoming_self_bucket[launch_eta] += sent_ships
source_mask[source_i]                     = False
```

`launch_eta`, source-target mesafesi ve `fleet_speed(sent_ships)` ile tahmin edilir. Böylece sonraki source kararında model, önceki launch'ın target üzerinde hangi ETA aralığında ne kadar pending saldırı/destek yarattığını görür.

Enemy incoming tarafı owner slotlarına ayrılmaz; tüm enemy filolar `incoming_enemy_0_5`, `incoming_enemy_5_15`, `incoming_enemy_15_30`, `incoming_enemy_30_plus` bucket'larında toplanır.

In [ ]:
torch.manual_seed(0)
net = PlanetOnlyOrbitWarsPolicy(d_model=64, n_heads=4, n_layers=2)
n_params = sum(p.numel() for p in net.parameters())
print(f'Toplam parametre: {n_params:,}')
print(f'model_variant={net.model_variant}, p_max={net.p_max}, f_max={net.f_max}')
print(net)

### 3.3 Forward pass

Birden fazla observation'ı `pack_observations` ile batch'leyebiliriz. Burada tek observation'ı 3'lü batch yapıp deniyoruz.

In [ ]:
batch = pack_observations([o, o, o])
out = net(**batch)
print(f'src_logits  : {tuple(out.src_logits.shape)}   (B, P_max+1)')
print(f'tgt_logits  : {tuple(out.tgt_logits.shape)}  (B, P_max, P_max)')
print(f'ship_logits : {tuple(out.ship_logits.shape)}  (B, P_max, n_ship_bins)')
print(f'value       : {tuple(out.value.shape)}             (B,)')
print(f'tokens      : {tuple(out.tokens.shape)}  (B, T, d_model)')

In [ ]:
out

### 3.4 Maskelemenin çalıştığını doğrulayalım

Padded gezegenlere ve geçersiz kaynaklara olasılık akmamalı.

In [ ]:
src_probs = torch.softmax(out.src_logits, dim=-1)[0]
tgt_probs = torch.softmax(out.tgt_logits, dim=-1)[0]
n = int(o.planet_mask.sum())
P = net.p_max

print(f'n_real_planets = {n} / P_max = {P}')
print(f'Toplam src_probs = {src_probs.sum().item():.4f}  (1.0 olmalı)')
if n < P:
    pad_src = src_probs[n:P].sum().item()
    print(f'Padded slot src kütlesi = {pad_src:.2e}')
print(f'NO-OP olasılığı            = {src_probs[P].item():.3f}')
invalid_own = src_probs[:n][~torch.from_numpy(o.source_mask[:n])].sum().item()
print(f'Geçersiz kaynak (own değil / boş) = {invalid_own:.2e}')

print()
diag = tgt_probs.diagonal(dim1=0, dim2=1)
print(f'Diyagonal (self-target) max prob: {diag.max().item():.2e}')
if n < P:
    pad_tgt = tgt_probs[:n, n:].max().item()
    print(f'Padded target max prob: {pad_tgt:.2e}')
else:
    print('(P_max dolu, padded target slot kalmadı.)')

Beklediğimiz gibi: padded ve yasadışı slotlara sızıntı yok (~`1e-7` veya daha düşük; `-inf` masking sayesinde).

### 3.5 Aksiyon örnekleme

`net.act(out)` her batch elemanı için bir aksiyon sampling yapar.

In [ ]:
torch.manual_seed(42)
action = net.act(out, deterministic=False)
for k, v in action.items():
    print(f'  {k:<10s} = {v.tolist()}')

## 4. Uçtan uca: obs → policy → env aksiyon

Policy'nin önerdiği `(src_slot, tgt_slot, ship_bin)` üçlüsünü `decode_action` ile env formatına (`[from_id, angle, ships]`) çeviriyoruz. `compute_launch` yörüngeyi ve açıyı çözer; geçerli yörünge yoksa `None` döner (NO-OP).

In [ ]:
planets = parse_planets(obs0)
src_slot = int(action['src'][0].item())
tgt_slot = int(action['tgt'][0].item())
ship_bin = int(action['ship_bin'][0].item())
is_noop = bool(action['is_noop'][0].item())
print(f'Policy seçimi: src_slot={src_slot}, tgt_slot={tgt_slot}, ship_bin={ship_bin}, noop={is_noop}')

env_action = decode_action(o, src_slot, tgt_slot, ship_bin,
                           planets=planets, angular_velocity=omega,
                           no_op=is_noop)
print(f'Env aksiyonu: {env_action}')
if env_action is not None:
    fid, angle, ships = env_action
    print(f'  -> from_planet={fid}, angle={math.degrees(angle):+.1f}°, ships={ships}')

### 4.1 Random policy ile bir tur oynat

Henüz eğitim yok, ağırlıklar random. Yine de pipeline'ın çalıştığını gözlemleyebiliriz.

In [ ]:
def policy_action(env_obs, player, omega, net):
    obs = build_observation(env_obs, player=player, angular_velocity=omega)
    batch = pack_observations([obs])
    out = net(**batch)
    a = net.act(out, deterministic=False)
    src = int(a['src'][0]); tgt = int(a['tgt'][0]); bin_ = int(a['ship_bin'][0])
    is_noop = bool(a['is_noop'][0])
    planets = parse_planets(env_obs)
    decoded = decode_action(obs, src, tgt, bin_, planets=planets,
                            angular_velocity=omega, no_op=is_noop)
    return [decoded] if decoded is not None else []  # env aksiyon listesi bekler

# Fresh env, prime the world with one empty step.
env.reset(2)
env.step([[], []])

for step in range(5):
    obs0 = env.state[0].observation
    obs1 = env.state[1].observation
    om = obs0.get('angular_velocity', 0.0)
    a0 = policy_action(obs0, 0, om, net)
    a1 = policy_action(obs1, 1, om, net)
    env.step([a0, a1])
    print(f'step {step+1}: p0 launched {len(a0)} fleet(s), p1 launched {len(a1)} fleet(s)')

final_obs = env.state[0].observation
print(f'\nFinal: planets={len(final_obs["planets"])}, fleets in flight={len(final_obs["fleets"])}')

## 5. Özet ve güncel inceleme notları

**Bu defterde gördüklerimiz:**

1. **`obs_builder.build_observation`** — raw env obs → padded numpy tensörler (`Observation` dataclass).
2. **31-d planet, 12-d global** default feature setleri; `planet_only` default'ta fleet token yoktur (`f_max=0`).
3. **Per-planet incoming ETA bucket aggregate** — her gezegen için self ve aggregate enemy incoming gemileri `0-5`, `5-15`, `15-30`, `30+` turn bucket'larında ve ek total/min/largest threat kanallarında planet token'a girer.
4. **Edge bias (T×T×5)** — transformer'a mesafe, yön (`dx, dy`) ve güneşe göre orbital açı farkı (`sin Δφ, cos Δφ`) prior olarak akıyor.
5. **Üç maske**: `planet_mask` (padding), `fleet_mask` (planet-only default'ta boş), `source_mask` (yasadışı kaynaklar).
6. **`PlanetOnlyOrbitWarsPolicy`** — planet + global token Set Transformer + masked pointer policy + value head.
7. **Güncel training rollout** — `act_source` kullanır; her owned source için ayrı launch/no-launch kararı verir. PPO tarafında `evaluate_source_actions` ile aynı log-prob semantiği tekrar hesaplanır.
8. **Action decode** — policy açı üretmez; `compute_launch` hareketli hedefe göre doğru radyan açıyı ve ETA’yı hesaplar.
9. **Saved training batch** — `--save-game-batches` açıkken her oyun `.pt` olarak saklanır. Planet-only batch’te `fleet_feats` şekli `(N, 0, 11)` olur.

**Son diagnostic notu:** Launch/no-op collapse incelenirken kod değiştirmeden rollout batch istatistikleri ölçüldü. Örnek checkpoint üzerinde launch transition’larının normalize advantage ortalaması no-op transition’larından düşük çıktı; aynı batch ile tek PPO update launch logitlerini düşürdü. Bu not model/batch incelemesi içindir; reward tasarımı kararı bu notebook’ta verilmez.

**Sonraki geliştirme fikirleri:**

- `ship_head`'i sadece source'a değil `(source, target)` çiftine koşullandırmak.
- Target skorunu `MLP([token_i, token_j, token_i * token_j, edge_ij, action_features_ij])` ile relation-centric yapmak.
- `compute_launch` reachability/ETA bilgisini target scoring’e explicit feature olarak vermek.
- ETA bucket sınırlarını ve reward bileşenlerini batch advantage istatistikleriyle ablation etmek.